# AMEX Enterprise Credit Risk Platform
## Notebook 44 -- Early Warning System: Validation & Deployment
### Phase 3 . Problem Statement 7: Early Warning System

CRISP-DM stage: **Evaluation & Deployment**. Sprint 1, Notebook 3 of 4 for this problem. Depends on Problem 1 Notebooks 01/02/05 and this problem's own Notebooks 42-43 (`early_warning_policy.json`, `early_warning_modeling_results.json`).

**What this notebook does (real, computed on your machine when you run it):**
- Deterministically rebuilds Notebook 43's rolling baseline/latest store (same reusable function, same inputs, no randomness anywhere) and reproduces its real per-customer `EARLY_WARNING_SCORE`, secondary ROC-AUC/PR-AUC, and base default rate as an explicit integrity check -- any mismatch is treated as a real bug, not sampling variation
- Selects the winning `MIN_DEVIATION_COUNT` candidate: the smallest (most sensitive) candidate that clears the real >=1.5x default-rate-lift KPI, or -- when none do -- the candidate with the real highest lift, explicitly flagged NOT RECOMMENDED FOR PRODUCTION, the same honest-selection pattern Notebooks 36/40 established for their own sweeps
- Runs 2,000-resample bootstrap confidence intervals for the PRIMARY KPI (default-rate lift at the winning candidate) as well as the secondary ROC-AUC/PR-AUC -- the first Validation & Deployment notebook in this platform to bootstrap the actual business KPI, not just a model-quality metric
- Runs an honest score-rank calibration check (does a higher score track a higher real default rate, monotonically -- the property an alerting system needs, not strict predicted-probability calibration) and a split-half Population Stability Index on the score distribution
- Assembles a full statistical validation table combining the reproduced metrics suite, both confidence intervals, calibration, and PSI
- Persists a deployment policy JSON (there is no trained model to persist -- this technique is stateless business logic) and generates `real_time_alert_service.py`, a genuinely different deployment shape from Problems 1/5/6's services: it scores a customer's full STATEMENT HISTORY (not a single flat feature row), computing the rolling baseline and z-scores server-side at request time
- Live-tests the generated service end to end against a REAL holdout customer's REAL statement history pulled fresh from the raw CSV, cross-checking the API's `EARLY_WARNING_SCORE` against this notebook's own precomputed value for that exact customer
- Benchmarks API latency, produces a deployment readiness checklist, and a Word validation & deployment report reusing Notebook 43's ROC/PR/lift charts directly

**What this notebook does NOT do:** it trains no model (there is none to train) and it does not decide the platform's final financial recommendation -- that, plus the elevated, multi-notebook-synthesizing Word/HTML reporting standard, is Notebook 45's job, closing out Problem 7.

Zero-fabrication: every number this notebook prints is computed live -- reproduced from real Notebook 43 data with zero randomness, or freshly bootstrap-resampled with the platform's standard `random_state=42`.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND PROBLEM 6/7'S REAL OUTPUTS (NOTEBOOKS 38, 40, 42, 43)
# =============================================================================
import os
import sys
import json
import time
import gc
import importlib
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 38, 40, 42, 43")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB42_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_42_summary.json"
NB43_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_43_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first"),
    (NB42_SUMMARY_PATH, "run 42_early_warning_system_business_understanding.ipynb first"),
    (NB43_SUMMARY_PATH, "run 43_early_warning_system_modeling.ipynb first "
                         "(this notebook selects a winning candidate from its real results)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB42_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB42_SUMMARY = json.load(f)
with open(NB43_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB43_SUMMARY = json.load(f)

EARLY_WARNING_POLICY_PATH = Path(NB42_SUMMARY["policy_path"])
with open(EARLY_WARNING_POLICY_PATH, "r", encoding="utf-8") as f:
    EARLY_WARNING_POLICY = json.load(f)

MODELING_RESULTS_PATH = Path(NB43_SUMMARY["modeling_results_path"])
if not MODELING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"{MODELING_RESULTS_PATH} not found.\nFix: re-run Notebook 43.")
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_RESULTS_ARTIFACT = json.load(f)

Z_THRESHOLD = EARLY_WARNING_POLICY["z_threshold"]
MIN_STATEMENTS_FOR_BASELINE = EARLY_WARNING_POLICY["min_statements_for_baseline"]
MIN_DEVIATION_COUNT_CANDIDATES = EARLY_WARNING_POLICY["min_deviation_count_candidates"]
CANDIDATE_FEATURES = EARLY_WARNING_POLICY["monitored_features"]["features"]
N_MONITORED_FEATURES = len(CANDIDATE_FEATURES)
EWS_KPI_TARGETS = EARLY_WARNING_POLICY["kpi_targets"]
CANDIDATE_RESULTS = {int(k): v for k, v in MODELING_RESULTS_ARTIFACT["candidate_results"].items()}
CANDIDATES_MEETING_KPI = [int(c) for c in MODELING_RESULTS_ARTIFACT["candidates_meeting_kpi"]]
NB43_ROC_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["roc_curve"])
NB43_PR_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["pr_curve"])
NB43_LIFT_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["lift_by_candidate"])

# --- Winning-candidate selection: same honest pattern Notebooks 36/40
#     established for their own K/W sweeps -- the SMALLEST (most sensitive,
#     broadest-net) candidate that still clears the real lift KPI, since a
#     lower MIN_DEVIATION_COUNT alerts more customers while still meeting
#     the quality bar; if NONE clear it, the candidate with the real HIGHEST
#     default-rate lift is selected instead and flagged NOT RECOMMENDED FOR
#     PRODUCTION -- packaged for completeness, not silently hidden. ---
if CANDIDATES_MEETING_KPI:
    WINNING_MIN_DEVIATION_COUNT = min(CANDIDATES_MEETING_KPI)
    MEETS_KPI = True
    print(f"Candidate(s) meeting the >= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift KPI: {CANDIDATES_MEETING_KPI}")
    print(f"Selected WINNING_MIN_DEVIATION_COUNT = {WINNING_MIN_DEVIATION_COUNT} "
          "(smallest/most-sensitive candidate that still clears the KPI)")
else:
    WINNING_MIN_DEVIATION_COUNT = max(
        CANDIDATE_RESULTS.keys(), key=lambda c: (CANDIDATE_RESULTS[c]["default_rate_lift"] or 0.0)
    )
    MEETS_KPI = False
    print(
        f"NONE of the real candidates {sorted(CANDIDATE_RESULTS.keys())} met the "
        f">= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift KPI on this real run. Selected "
        f"WINNING_MIN_DEVIATION_COUNT = {WINNING_MIN_DEVIATION_COUNT} (highest real default-rate lift among "
        f"candidates, {CANDIDATE_RESULTS[WINNING_MIN_DEVIATION_COUNT]['default_rate_lift']}) so this notebook "
        "can still validate and package a deployable service -- flagged NOT RECOMMENDED FOR PRODUCTION "
        "throughout, per the platform's zero-fabrication policy."
    )

WINNING_CANDIDATE_RESULT = CANDIDATE_RESULTS[WINNING_MIN_DEVIATION_COUNT]
print(f"\nWINNING_MIN_DEVIATION_COUNT = {WINNING_MIN_DEVIATION_COUNT}")
print(json.dumps(WINNING_CANDIDATE_RESULT, indent=2))

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "early_warning_models" in PILLAR_DIRS:
    EWS_MODELS_DIR = PILLAR_DIRS["early_warning_models"]
else:
    EWS_MODELS_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem7_Early_Warning_System" / "models"
    )
EWS_MODELS_DIR.mkdir(parents=True, exist_ok=True)

if "early_warning_reports" in PILLAR_DIRS:
    EWS_REPORTS_DIR = PILLAR_DIRS["early_warning_reports"]
else:
    EWS_REPORTS_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem7_Early_Warning_System" / "reports"
    )
EWS_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

EWS_API_DIR = (
    PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem7_Early_Warning_System" / "src" / "api"
)
EWS_API_DIR.mkdir(parents=True, exist_ok=True)
EWS_DEPLOYMENT_DIR = (
    PROJECT_ROOT / "Phase3_Behavioral_Intelligence" / "Problem7_Early_Warning_System" / "deployment"
)
EWS_DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nModels will be written under    : {EWS_MODELS_DIR}")
print(f"Reports will be written under   : {EWS_REPORTS_DIR}")
print(f"API service will be written under: {EWS_API_DIR}")
print(f"Deployment artifacts under      : {EWS_DEPLOYMENT_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve,
        confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import importlib.metadata as importlib_metadata
except ImportError:
    import importlib_metadata

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA RE-VERIFICATION -- MONITORED COLUMNS STILL PRESENT
# =============================================================================
_section("SECTION 4: Live Schema Re-Verification -- Monitored Columns Still Present")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)
_missing_cols = set(CANDIDATE_FEATURES) - _header_cols
if _missing_cols:
    raise RuntimeError(f"{len(_missing_cols)} monitored column(s) missing from the live raw CSV header: "
                        f"{sorted(_missing_cols)}")
print(f"Confirmed all {N_MONITORED_FEATURES} monitored columns are present in the live raw CSV header.")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: ROLLING BASELINE / LATEST-STATEMENT FEATURE ENGINEERING --
#            REUSABLE FUNCTION (REUSED VERBATIM FROM NOTEBOOK 43)
# =============================================================================
_section("SECTION 5: Rolling Baseline / Latest-Statement Feature Engineering -- Reusable Function")


def build_rolling_zscore_store(csv_path: Path, base_cols: list, min_statements: int) -> "pl.DataFrame":
    """Identical to Notebook 43's function -- see there for the full docstring.
    Reused verbatim (not re-derived) so this notebook's reproduction of
    Notebook 43's numbers is a genuine reproducibility check, not a
    re-implementation that could silently diverge."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_n_statements") >= min_statements)
    )
    _is_baseline = pl.col("_row_idx") < (pl.col("_n_statements") - 1)

    agg_exprs = [pl.first("_n_statements").alias("n_statements")]
    for c in base_cols:
        _baseline_val = pl.when(_is_baseline).then(pl.col(c)).otherwise(None)
        agg_exprs += [
            _baseline_val.mean().alias(f"_baseline_mean_{c}"),
            _baseline_val.std(ddof=1).alias(f"_baseline_std_{c}"),
            _baseline_val.count().alias(f"_baseline_n_{c}"),
            pl.col(c).last().alias(f"_latest_{c}"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    return grouped.sort("customer_ID").collect(engine="streaming")


print("build_rolling_zscore_store() defined (reused verbatim from Notebook 43).")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: LOAD LABELS & TRAIN/VALIDATION SPLIT MEMBERSHIP
# =============================================================================
_section("SECTION 6: Load Labels & Train/Validation Split Membership")

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")
print(f"Train-split customers (from Notebook 02, reused)     : {len(train_ids_set):,}")
print(f"Validation-split customers (from Notebook 02, reused): {len(val_ids_set):,}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: REBUILD & REPRODUCE NOTEBOOK 43's REAL RESULTS (INTEGRITY CHECK)
# =============================================================================
_section("SECTION 7: Rebuild & Reproduce Notebook 43's Real Results (Integrity Check)")

print(
    "This deterministically rebuilds the same rolling baseline/latest store Notebook 43 built (same "
    "function, same inputs, same random seed -- there is no randomness in this computation at all) and "
    "recomputes every per-customer EARLY_WARNING_SCORE from scratch, then cross-checks the result against "
    "Notebook 43's persisted numbers -- a genuine reproducibility check, not a re-implementation that could "
    "silently diverge. This also regenerates the per-customer arrays this notebook needs for bootstrap "
    "confidence intervals, calibration, and PSI (Notebook 43 persisted only aggregate summaries)."
)

gc.collect()
_t0 = time.time()
zscore_store = build_rolling_zscore_store(RAW_TRAIN_DATA_PATH, CANDIDATE_FEATURES, MIN_STATEMENTS_FOR_BASELINE)
print(f"Rebuilt store for {zscore_store.height:,} eligible customers in {time.time() - _t0:.1f}s. "
      f"Process RSS: {_rss_gb():.2f} GB")

engineered = zscore_store.join(labels_df, on="customer_ID", how="inner")
del zscore_store
gc.collect()
holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
del engineered
gc.collect()

_mean_cols = [f"_baseline_mean_{c}" for c in CANDIDATE_FEATURES]
_std_cols = [f"_baseline_std_{c}" for c in CANDIDATE_FEATURES]
_n_cols = [f"_baseline_n_{c}" for c in CANDIDATE_FEATURES]
_latest_cols = [f"_latest_{c}" for c in CANDIDATE_FEATURES]

mean_arr = holdout_df.select(_mean_cols).to_numpy().astype(np.float64)
std_arr = holdout_df.select(_std_cols).to_numpy().astype(np.float64)
n_arr = holdout_df.select(_n_cols).to_numpy().astype(np.float64)
latest_arr = holdout_df.select(_latest_cols).to_numpy().astype(np.float64)
y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64)
holdout_customer_ids = holdout_df.get_column("customer_ID").to_numpy()
del holdout_df
gc.collect()

with np.errstate(invalid="ignore", divide="ignore"):
    z_scores = (latest_arr - mean_arr) / std_arr
z_computable_mask = (n_arr >= 2) & (std_arr > 0) & ~np.isnan(latest_arr) & ~np.isnan(mean_arr) & ~np.isnan(std_arr)
z_scores = np.where(z_computable_mask, z_scores, np.nan)
deviates_mask = np.where(z_computable_mask, np.abs(z_scores) >= Z_THRESHOLD, False)
EARLY_WARNING_SCORE = deviates_mask.sum(axis=1).astype(np.int64)
EARLY_WARNING_SCORE_NORMALIZED = EARLY_WARNING_SCORE.astype(np.float64) / N_MONITORED_FEATURES

reproduced_base_default_rate = float(y_holdout.mean())
reproduced_auc = float(roc_auc_score(y_holdout, EARLY_WARNING_SCORE_NORMALIZED))
reproduced_pr_auc = float(average_precision_score(y_holdout, EARLY_WARNING_SCORE_NORMALIZED))
_eps = 1e-7
reproduced_log_loss = float(
    log_loss(y_holdout, np.clip(EARLY_WARNING_SCORE_NORMALIZED, _eps, 1.0 - _eps), labels=[0, 1])
)
fpr, tpr, _ = roc_curve(y_holdout, EARLY_WARNING_SCORE_NORMALIZED)
pr_precision, pr_recall, _ = precision_recall_curve(y_holdout, EARLY_WARNING_SCORE_NORMALIZED)

_reported = MODELING_RESULTS_ARTIFACT["secondary_threshold_free_metrics"]
print(f"Reproduced holdout customers scored: {len(y_holdout):,}  (Notebook 43 reported "
      f"{MODELING_RESULTS_ARTIFACT['n_holdout_customers_scored']:,})")
print(f"Reproduced base default rate: {reproduced_base_default_rate:.4f}  "
      f"(Notebook 43 reported {MODELING_RESULTS_ARTIFACT['base_default_rate_holdout']:.4f})")
print(f"Reproduced ROC-AUC : {reproduced_auc:.4f}  (Notebook 43 reported {_reported['roc_auc']:.4f})")
print(f"Reproduced PR-AUC  : {reproduced_pr_auc:.4f}  (Notebook 43 reported {_reported['pr_auc']:.4f})")
print(f"Reproduced LogLoss : {reproduced_log_loss:.4f}  (Notebook 43 reported {_reported['log_loss']:.4f})")

_reproduction_matches = (
    len(y_holdout) == MODELING_RESULTS_ARTIFACT["n_holdout_customers_scored"]
    and abs(reproduced_auc - _reported["roc_auc"]) < 1e-6
    and abs(reproduced_pr_auc - _reported["pr_auc"]) < 1e-6
)
print(f"\nReproduction matches Notebook 43 exactly (deterministic, same data): {_reproduction_matches}")
if not _reproduction_matches:
    raise RuntimeError("Notebook 44's reproduction does NOT match Notebook 43's persisted numbers -- "
                        "investigate before proceeding (this computation has no randomness, so any "
                        "mismatch indicates a real bug, not sampling variation).")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: REPRODUCE THE WINNING CANDIDATE'S FULL METRICS SUITE
# =============================================================================
_section("SECTION 8: Reproduce the Winning Candidate's Full Metrics Suite")

_pred = (EARLY_WARNING_SCORE >= WINNING_MIN_DEVIATION_COUNT).astype(np.int64)
_tn, _fp, _fn, _tp = confusion_matrix(y_holdout, _pred, labels=[0, 1]).ravel()
_specificity = float(_tn / (_tn + _fp)) if (_tn + _fp) > 0 else 0.0
_n_alerted = int(_pred.sum())
_default_rate_alerted = float(y_holdout[_pred == 1].mean()) if _n_alerted > 0 else None
_lift = (
    (_default_rate_alerted / reproduced_base_default_rate)
    if (_default_rate_alerted is not None and reproduced_base_default_rate > 0) else None
)
winning_metrics = {
    "candidate_min_deviation_count": int(WINNING_MIN_DEVIATION_COUNT),
    "n_alerted": _n_alerted,
    "pct_alerted": 100.0 * _n_alerted / len(y_holdout),
    "default_rate_alerted": _default_rate_alerted,
    "default_rate_lift": _lift,
    "meets_kpi_target": bool(_lift is not None and _lift >= EWS_KPI_TARGETS["min_default_rate_lift"]),
    "accuracy": float(accuracy_score(y_holdout, _pred)),
    "precision": float(precision_score(y_holdout, _pred, zero_division=0)),
    "recall": float(recall_score(y_holdout, _pred, zero_division=0)),
    "f1": float(f1_score(y_holdout, _pred, zero_division=0)),
    "specificity": _specificity,
    "mcc": float(matthews_corrcoef(y_holdout, _pred)),
    "confusion_matrix": {"tn": int(_tn), "fp": int(_fp), "fn": int(_fn), "tp": int(_tp)},
}
_winning_reproduction_matches = (
    abs((winning_metrics["default_rate_lift"] or 0.0) - (WINNING_CANDIDATE_RESULT["default_rate_lift"] or 0.0)) < 1e-9
    and winning_metrics["confusion_matrix"] == WINNING_CANDIDATE_RESULT["confusion_matrix"]
)
print(json.dumps(winning_metrics, indent=2))
print(f"\nReproduction matches Notebook 43's per-candidate record for candidate={WINNING_MIN_DEVIATION_COUNT}: "
      f"{_winning_reproduction_matches}")
if not _winning_reproduction_matches:
    raise RuntimeError("Winning candidate's reproduced confusion matrix/lift does not match Notebook 43 -- "
                        "investigate before proceeding.")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: BOOTSTRAP CONFIDENCE INTERVALS -- DEFAULT-RATE LIFT (PRIMARY),
#            ROC-AUC & PR-AUC (SECONDARY)
# =============================================================================
_section("SECTION 9: Bootstrap Confidence Intervals -- Lift (Primary) and AUC/PR-AUC (Secondary)")

N_BOOTSTRAP = 2000
_rng = np.random.default_rng(RANDOM_SEED)
_n_holdout = len(y_holdout)

_boot_lifts = np.empty(N_BOOTSTRAP, dtype=np.float64)
_boot_aucs = np.empty(N_BOOTSTRAP, dtype=np.float64)
_boot_pr_aucs = np.empty(N_BOOTSTRAP, dtype=np.float64)
for _b in range(N_BOOTSTRAP):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _yt = y_holdout[_idx]
    _score_boot = EARLY_WARNING_SCORE[_idx]
    _score_norm_boot = EARLY_WARNING_SCORE_NORMALIZED[_idx]
    _pred_boot = (_score_boot >= WINNING_MIN_DEVIATION_COUNT)
    _n_alert_boot = int(_pred_boot.sum())
    _base_rate_boot = float(_yt.mean())
    if _n_alert_boot > 0 and _base_rate_boot > 0:
        _boot_lifts[_b] = float(_yt[_pred_boot].mean()) / _base_rate_boot
    else:
        _boot_lifts[_b] = np.nan
    if _yt.min() == _yt.max():
        _boot_aucs[_b] = np.nan  # degenerate resample (all-one-class); excluded below
        _boot_pr_aucs[_b] = np.nan
    else:
        _boot_aucs[_b] = roc_auc_score(_yt, _score_norm_boot)
        _boot_pr_aucs[_b] = average_precision_score(_yt, _score_norm_boot)

_valid_lift_boots = _boot_lifts[~np.isnan(_boot_lifts)]
_valid_auc_boots = _boot_aucs[~np.isnan(_boot_aucs)]
_valid_pr_boots = _boot_pr_aucs[~np.isnan(_boot_pr_aucs)]
LIFT_CI_LOWER, LIFT_CI_UPPER = np.percentile(_valid_lift_boots, [2.5, 97.5])
AUC_CI_LOWER, AUC_CI_UPPER = np.percentile(_valid_auc_boots, [2.5, 97.5])
PR_AUC_CI_LOWER, PR_AUC_CI_UPPER = np.percentile(_valid_pr_boots, [2.5, 97.5])

print(f"Bootstrap resamples: {N_BOOTSTRAP:,} (valid lift resamples: {len(_valid_lift_boots):,}, "
      f"random_state={RANDOM_SEED})")
print(f"Default-rate lift @ candidate={WINNING_MIN_DEVIATION_COUNT} 95% CI: "
      f"[{LIFT_CI_LOWER:.3f}x, {LIFT_CI_UPPER:.3f}x]  (point estimate "
      f"{(winning_metrics['default_rate_lift'] or 0.0):.3f}x)")
print(f"ROC-AUC 95% CI   : [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}]  (point estimate {reproduced_auc:.4f})")
print(f"PR-AUC 95% CI    : [{PR_AUC_CI_LOWER:.4f}, {PR_AUC_CI_UPPER:.4f}]  (point estimate {reproduced_pr_auc:.4f}, "
      f"no-skill baseline {reproduced_base_default_rate:.4f})")
_lift_ci_excludes_no_lift = LIFT_CI_LOWER > 1.0
_ci_excludes_random = AUC_CI_LOWER > 0.5
_pr_ci_excludes_noskill = PR_AUC_CI_LOWER > reproduced_base_default_rate
print(f"Lift 95% CI entirely above 1.0x (a real, non-chance concentration of defaulters among alerted "
      f"customers): {_lift_ci_excludes_no_lift}")
print(f"AUC 95% CI entirely above random (0.5): {_ci_excludes_random}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: CALIBRATION CHECK -- NORMALIZED SCORE BINS VS. REAL OBSERVED
#             DEFAULT RATE (INFORMATIONAL)
# =============================================================================
_section("SECTION 10: Calibration Check (Informational)")

print(
    "Informational only, not a KPI gate: EARLY_WARNING_SCORE was never fit to BE a calibrated probability "
    "(unlike Problem 6's trained model) -- this check simply asks whether a HIGHER normalized score tracks "
    "a HIGHER real observed default rate, monotonically, which is the property an alerting system actually "
    "needs (ranking), not calibration in the strict predicted-probability sense."
)

_calib_df = pd.DataFrame({"score": EARLY_WARNING_SCORE_NORMALIZED, "target": y_holdout})
try:
    _calib_df["bin"] = pd.qcut(_calib_df["score"], q=5, labels=False, duplicates="drop")
except ValueError:
    _calib_df["bin"] = pd.cut(_calib_df["score"], bins=5, labels=False, duplicates="drop")
calibration_table = (
    _calib_df.groupby("bin")
    .agg(n=("target", "size"), mean_score=("score", "mean"), observed_default_rate=("target", "mean"))
    .reset_index()
    .sort_values("mean_score")
)
print(calibration_table.round(4).to_string(index=False))
_rates = calibration_table["observed_default_rate"].to_numpy()
CALIBRATION_MONOTONIC = bool(np.all(np.diff(_rates) >= -1e-9))
print(f"\nObserved default rate is monotonically non-decreasing across score bins (real, measured): "
      f"{CALIBRATION_MONOTONIC}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: SPLIT-HALF POPULATION STABILITY (PSI) ON THE NORMALIZED SCORE
# =============================================================================
_section("SECTION 11: Split-Half Population Stability (PSI) on the Normalized Score")

# --- Same honest framing this platform already documents for Notebook 21/
#     36/40's own PSI: a random split-half stability proxy on the single
#     available holdout population, not a genuine time-based drift
#     measurement -- there is no second real time period to compare against
#     yet. Bin edges come from the full holdout score distribution's real
#     deciles, applied to two random halves of that same population. ---
_score_edges = np.quantile(EARLY_WARNING_SCORE_NORMALIZED, np.linspace(0, 1, 11))
_score_edges = np.unique(_score_edges)
if len(_score_edges) < 3:
    _score_edges = np.array([-np.inf, np.median(EARLY_WARNING_SCORE_NORMALIZED), np.inf])
else:
    _score_edges[0], _score_edges[-1] = -np.inf, np.inf
_perm = _rng.permutation(_n_holdout)
_half = _n_holdout // 2
_half_a = EARLY_WARNING_SCORE_NORMALIZED[_perm[:_half]]
_half_b = EARLY_WARNING_SCORE_NORMALIZED[_perm[_half:]]
_share_a = np.histogram(_half_a, bins=_score_edges)[0] / len(_half_a)
_share_b = np.histogram(_half_b, bins=_score_edges)[0] / len(_half_b)
_share_a = np.clip(_share_a, 1e-4, None)
_share_b = np.clip(_share_b, 1e-4, None)
SCORE_PSI_SPLIT_HALF = float(((_share_a - _share_b) * np.log(_share_a / _share_b)).sum())
_psi_target = 0.10  # ASSUMPTION: standard industry PSI stability threshold (<0.10 = no significant shift)
print(f"Split-half PSI on the normalized EARLY_WARNING_SCORE: {SCORE_PSI_SPLIT_HALF:.4f}  (target < {_psi_target}, "
      f"{'PASS' if SCORE_PSI_SPLIT_HALF < _psi_target else 'FAIL'})")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: FULL METRICS-SUITE STATISTICAL VALIDATION SUMMARY TABLE
# =============================================================================
_section("SECTION 12: Full Metrics-Suite Statistical Validation Summary Table")

_random_baseline_log_loss = 0.6931  # ln(2): log loss of a constant 0.5 prediction
statistical_validation_rows = [
    {"test": f"Default-rate lift @ candidate={WINNING_MIN_DEVIATION_COUNT} (reproduced)",
     "value": round(winning_metrics["default_rate_lift"] or 0.0, 3),
     "target": f">={EWS_KPI_TARGETS['min_default_rate_lift']}x", "pass": bool(MEETS_KPI)},
    {"test": "Lift 95% CI lower bound", "value": round(float(LIFT_CI_LOWER), 3), "target": ">1.0x",
     "pass": bool(_lift_ci_excludes_no_lift)},
    {"test": "ROC-AUC (reproduced, secondary)", "value": round(reproduced_auc, 4), "target": ">0.5 (reported)",
     "pass": bool(reproduced_auc > 0.5)},
    {"test": "ROC-AUC 95% CI lower bound", "value": round(float(AUC_CI_LOWER), 4), "target": ">0.5 (reported)",
     "pass": bool(_ci_excludes_random)},
    {"test": "PR-AUC (reproduced, secondary)", "value": round(reproduced_pr_auc, 4),
     "target": f">{reproduced_base_default_rate:.4f} (no-skill, reported)",
     "pass": bool(reproduced_pr_auc > reproduced_base_default_rate)},
    {"test": "Log Loss (reproduced, secondary)", "value": round(reproduced_log_loss, 4),
     "target": f"<{_random_baseline_log_loss} (reported)", "pass": bool(reproduced_log_loss < _random_baseline_log_loss)},
    {"test": f"Accuracy @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["accuracy"], 4),
     "target": "reported", "pass": True},
    {"test": f"Precision @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["precision"], 4),
     "target": "reported", "pass": True},
    {"test": f"Recall @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["recall"], 4),
     "target": "reported", "pass": True},
    {"test": f"F1 @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["f1"], 4),
     "target": "reported", "pass": True},
    {"test": f"Specificity @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["specificity"], 4),
     "target": "reported", "pass": True},
    {"test": f"MCC @ candidate={WINNING_MIN_DEVIATION_COUNT}", "value": round(winning_metrics["mcc"], 4),
     "target": "reported", "pass": True},
    {"test": "Score-rank calibration monotonicity", "value": CALIBRATION_MONOTONIC, "target": "True",
     "pass": bool(CALIBRATION_MONOTONIC)},
    {"test": "Split-half score PSI", "value": round(SCORE_PSI_SPLIT_HALF, 4), "target": f"<{_psi_target}",
     "pass": bool(SCORE_PSI_SPLIT_HALF < _psi_target)},
    {"test": "Default-rate lift KPI (Notebook 42 target)", "value": round(winning_metrics["default_rate_lift"] or 0.0, 3),
     "target": f">={EWS_KPI_TARGETS['min_default_rate_lift']}x", "pass": bool(MEETS_KPI)},
]
statistical_validation_df = pd.DataFrame(statistical_validation_rows)
statistical_validation_path = EWS_DEPLOYMENT_DIR / "early_warning_statistical_validation.csv"
statistical_validation_df.to_csv(statistical_validation_path, index=False)
print(statistical_validation_df.to_string(index=False))
ALL_STAT_CHECKS_PASS = bool(statistical_validation_df["pass"].all())
print(f"\nAll statistical checks pass: {ALL_STAT_CHECKS_PASS}")
print(f"✅ Saved -> {statistical_validation_path}")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 13: Honest Limitation -- Deployment Scope & Assumptions")

DEPLOYMENT_LIMITATION = {
    "technique_scope": (
        "This is an unsupervised, rule-based statistical-process-control technique, NOT a trained "
        "classifier -- it flags a customer whose LATEST statement deviates from THEIR OWN recent baseline "
        "in enough monitored features at once. It complements Problem 6's trained recency model; it is not "
        "a replacement for it or for Problem 1's full-history champion."
    ),
    "kpi_status": (
        f"MEETS the >={EWS_KPI_TARGETS['min_default_rate_lift']}x default-rate lift KPI set in Notebook 42."
        if MEETS_KPI else
        f"DOES NOT MEET the >={EWS_KPI_TARGETS['min_default_rate_lift']}x default-rate lift KPI set in "
        "Notebook 42 -- this is the best-performing candidate tested (highest real lift), packaged here for "
        "completeness and so validation/deployment tooling exists, but it is NOT RECOMMENDED FOR PRODUCTION "
        "USE until a future run either finds a viable candidate or the KPI target itself is revisited."
    ),
    "baseline_eligibility_limitation": (
        f"Only customers with >= {MIN_STATEMENTS_FOR_BASELINE} real statements can be monitored at all "
        f"(Notebook 42 measured {EARLY_WARNING_POLICY['baseline_eligibility_coverage_pct']:.1f}% real "
        "coverage) -- newer customers with a short history are honestly out of scope for this technique, "
        "not silently scored with a degenerate baseline."
    ),
    "small_alert_population_caveat": (
        f"At the winning candidate (MIN_DEVIATION_COUNT={WINNING_MIN_DEVIATION_COUNT}), only "
        f"{winning_metrics['n_alerted']:,} of {_n_holdout:,} holdout customers ({winning_metrics['pct_alerted']:.2f}%) "
        "are alerted -- the lift point estimate and its bootstrap CI above should both be read with this "
        "real sample size in mind; a small alerted population makes the lift estimate more sensitive to "
        "individual customers, which is exactly why the bootstrap CI (not just the point estimate) is "
        "reported."
    ),
    "feature_space_reuse_note": (
        "Reuses Problem 4/6's real, correlation-filtered base feature list (not a fresh selection) -- see "
        "Notebook 42 Section 4 for the recovery method."
    ),
}
for _k, _v in DEPLOYMENT_LIMITATION.items():
    print(f"{_k}:\n  {_v}\n")
print("\n✅ Section 13 complete.")


# =============================================================================
# SECTION 14: PERSIST POLICY & VALIDATION ARTIFACTS
# =============================================================================
_section("SECTION 14: Persist Policy & Validation Artifacts")

# --- There is no trained model to persist (this technique is stateless,
#     rule-based business logic) -- what gets persisted is the DEPLOYMENT
#     POLICY: the exact Z_THRESHOLD, monitored feature list, and winning
#     alert threshold a real-time service needs to reproduce this notebook's
#     computation on a NEW customer's statement history at inference time. ---
DEPLOYMENT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "z_threshold": Z_THRESHOLD,
    "min_statements_for_baseline": MIN_STATEMENTS_FOR_BASELINE,
    "winning_min_deviation_count": int(WINNING_MIN_DEVIATION_COUNT),
    "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "monitored_features": CANDIDATE_FEATURES,
    "winning_candidate_metrics": winning_metrics,
    "random_seed": RANDOM_SEED,
}
deployment_policy_path = EWS_MODELS_DIR / "early_warning_deployment_policy.json"
with open(deployment_policy_path, "w", encoding="utf-8") as f:
    json.dump(DEPLOYMENT_POLICY, f, indent=2)
print(f"✅ Saved -> {deployment_policy_path} ({deployment_policy_path.stat().st_size / 1e3:.1f} KB)")
print("\n✅ Section 14 complete.")


# =============================================================================
# SECTION 15: GENERATE real_time_alert_service.py -- REAL, RUNNABLE FASTAPI SERVICE
# =============================================================================
_section("SECTION 15: Generate real_time_alert_service.py -- Real FastAPI Service")

# --- Genuinely different deployment shape from Problems 1/5/6's services:
#     this technique needs a customer's STATEMENT HISTORY (a list of past
#     records), not a single flat feature row, since the baseline is
#     computed from the customer's own history at scoring time. Same
#     plain string-list generation pattern Notebooks 10/22/36/40
#     established (avoids f-string brace-escaping on the generated source's
#     own literal braces), same policy-JSON-driven, env-var-overridable
#     config convention. ---
_deployment_policy_path_str = str(deployment_policy_path)

REAL_TIME_ALERT_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Early Warning System Real-Time Alert API.",
    "# Auto-generated by 44_early_warning_system_validation_deployment.ipynb.",
    "# Unlike Problems 1/5/6's services, this one scores a CUSTOMER'S STATEMENT HISTORY (a list of past",
    "# records), not a single flat feature row -- the baseline is computed from that history at scoring",
    "# time, exactly reproducing Notebooks 43/44's rolling z-score computation.",
    "# Run with:",
    "#     uvicorn real_time_alert_service:app --host 0.0.0.0 --port 8004",
    "import json",
    "import os",
    "from pathlib import Path",
    "from typing import Dict, List, Optional",
    "",
    "import numpy as np",
    "from fastapi import FastAPI, HTTPException",
    "from pydantic import BaseModel",
    "",
    "POLICY_PATH = Path(os.environ.get(\"AMEX_EWS_POLICY_PATH\", r\"__POLICY_PATH_TOKEN__\"))",
    "with open(POLICY_PATH, \"r\", encoding=\"utf-8\") as _f:",
    "    _POLICY = json.load(_f)",
    "",
    "Z_THRESHOLD = _POLICY[\"z_threshold\"]",
    "MIN_STATEMENTS_FOR_BASELINE = _POLICY[\"min_statements_for_baseline\"]",
    "WINNING_MIN_DEVIATION_COUNT = _POLICY[\"winning_min_deviation_count\"]",
    "MONITORED_FEATURES = _POLICY[\"monitored_features\"]",
    "RECOMMENDED_FOR_PRODUCTION = _POLICY[\"recommended_for_production\"]",
    "",
    "",
    "class ScoreRequest(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    # Chronological order, OLDEST first, LAST item = the latest statement being tested.",
    "    # Each statement is a dict of {monitored_feature_name: value_or_null}.",
    "    statements: List[Dict[str, Optional[float]]]",
    "",
    "",
    "class AlertResponse(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    early_warning_score: int",
    "    monitored_feature_count: int",
    "    z_computable_feature_count: int",
    "    alert: bool",
    "    winning_min_deviation_count: int",
    "    feature_deviations: Dict[str, Optional[float]]",
    "",
    "",
    "def compute_early_warning(statements: List[Dict[str, Optional[float]]]) -> dict:",
    "    if len(statements) < MIN_STATEMENTS_FOR_BASELINE:",
    "        raise ValueError(",
    "            f\"Need >= {MIN_STATEMENTS_FOR_BASELINE} statements to form a baseline + latest, got \"",
    "            f\"{len(statements)}.\"",
    "        )",
    "    baseline_statements = statements[:-1]",
    "    latest_statement = statements[-1]",
    "    feature_deviations: Dict[str, Optional[float]] = {}",
    "    n_deviating = 0",
    "    n_computable = 0",
    "    for feat in MONITORED_FEATURES:",
    "        baseline_vals = [s[feat] for s in baseline_statements if s.get(feat) is not None]",
    "        latest_val = latest_statement.get(feat)",
    "        if len(baseline_vals) < 2 or latest_val is None:",
    "            feature_deviations[feat] = None",
    "            continue",
    "        arr = np.asarray(baseline_vals, dtype=np.float64)",
    "        baseline_mean = float(arr.mean())",
    "        baseline_std = float(arr.std(ddof=1))",
    "        if baseline_std <= 0:",
    "            feature_deviations[feat] = None",
    "            continue",
    "        z = (float(latest_val) - baseline_mean) / baseline_std",
    "        feature_deviations[feat] = z",
    "        n_computable += 1",
    "        if abs(z) >= Z_THRESHOLD:",
    "            n_deviating += 1",
    "    return {",
    "        \"early_warning_score\": n_deviating,",
    "        \"z_computable_feature_count\": n_computable,",
    "        \"feature_deviations\": feature_deviations,",
    "    }",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Early Warning System Real-Time Alert API\",",
    "    description=\"Flags a customer whose LATEST statement deviates from THEIR OWN recent baseline in \"",
    "                \"enough monitored features at once -- an unsupervised, rule-based control-chart \"",
    "                \"technique, complementary to Problem 6's trained recency model. See /model-info for \"",
    "                \"the real validation metrics behind this policy.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"winning_min_deviation_count\": WINNING_MIN_DEVIATION_COUNT}",
    "",
    "",
    "@app.get(\"/model-info\")",
    "def model_info():",
    "    return {",
    "        \"z_threshold\": Z_THRESHOLD,",
    "        \"min_statements_for_baseline\": MIN_STATEMENTS_FOR_BASELINE,",
    "        \"winning_min_deviation_count\": WINNING_MIN_DEVIATION_COUNT,",
    "        \"monitored_feature_count\": len(MONITORED_FEATURES),",
    "        \"winning_candidate_metrics\": _POLICY[\"winning_candidate_metrics\"],",
    "        \"recommended_for_production\": RECOMMENDED_FOR_PRODUCTION,",
    "    }",
    "",
    "",
    "@app.post(\"/score\", response_model=AlertResponse)",
    "def score(request: ScoreRequest):",
    "    try:",
    "        result = compute_early_warning(request.statements)",
    "    except ValueError as exc:",
    "        raise HTTPException(status_code=422, detail=str(exc))",
    "    except Exception as exc:",
    "        raise HTTPException(status_code=500, detail=\"Scoring failed: \" + str(exc))",
    "    alert = result[\"early_warning_score\"] >= WINNING_MIN_DEVIATION_COUNT",
    "    return AlertResponse(",
    "        customer_id=request.customer_id,",
    "        early_warning_score=result[\"early_warning_score\"],",
    "        monitored_feature_count=len(MONITORED_FEATURES),",
    "        z_computable_feature_count=result[\"z_computable_feature_count\"],",
    "        alert=alert,",
    "        winning_min_deviation_count=WINNING_MIN_DEVIATION_COUNT,",
    "        feature_deviations=result[\"feature_deviations\"],",
    "    )",
    "",
])
REAL_TIME_ALERT_SERVICE_SOURCE = REAL_TIME_ALERT_SERVICE_TEMPLATE.replace(
    "__POLICY_PATH_TOKEN__", _deployment_policy_path_str
)

service_py_path = EWS_API_DIR / "real_time_alert_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(REAL_TIME_ALERT_SERVICE_SOURCE)
compile(REAL_TIME_ALERT_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(REAL_TIME_ALERT_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"✅ Saved -> {service_py_path}")
print("\n✅ Section 15 complete.")


# =============================================================================
# SECTION 16: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 16: Generate .env.example & requirements-api.txt")

ENV_EXAMPLE = f"""# Copy to .env and edit if this machine's policy file location differs from the default.
AMEX_EWS_POLICY_PATH={deployment_policy_path}
"""
env_example_path = EWS_API_DIR / ".env.example"
with open(env_example_path, "w", encoding="utf-8") as f:
    f.write(ENV_EXAMPLE)

_api_packages = ["fastapi", "uvicorn", "pydantic", "numpy"]
_api_pkg_versions = {}
for _pkg in _api_packages:
    try:
        _api_pkg_versions[_pkg] = importlib_metadata.version(_pkg)
    except importlib_metadata.PackageNotFoundError:
        _api_pkg_versions[_pkg] = None

requirements_api_path = EWS_API_DIR / "requirements-api.txt"
with open(requirements_api_path, "w", encoding="utf-8") as f:
    f.write(f"# Minimal runtime dependencies for real_time_alert_service.py -- auto-generated "
             f"{datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    for _pkg, _ver in _api_pkg_versions.items():
        f.write(f"{_pkg}=={_ver}\n" if _ver else f"# {_pkg}  -- not installed here\n")

print(f"✅ Saved -> {env_example_path}")
print(f"✅ Saved -> {requirements_api_path}")
print("\n✅ Section 16 complete.")


# =============================================================================
# SECTION 17: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT WITH
#             A REAL CUSTOMER'S ACTUAL STATEMENT HISTORY
# =============================================================================
_section("SECTION 17: Live Self-Test -- Import the Generated Service & Drive It")

# --- Imports the EXACT file just written to disk -- proves the delivered
#     artifact works, not just an in-notebook copy of the same logic. Drives
#     it with a REAL holdout customer's REAL statement history pulled fresh
#     from the raw CSV, and cross-checks the API's early_warning_score
#     against this notebook's own precomputed value for that same customer
#     (Section 7/8) -- a genuine end-to-end consistency check. ---
os.environ["AMEX_EWS_POLICY_PATH"] = str(deployment_policy_path)
_spec = importlib.util.spec_from_file_location("amex_real_time_alert_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)

_health_resp = client.get("/health")
assert _health_resp.status_code == 200, f"/health returned {_health_resp.status_code}"
print(f"GET /health     -> {_health_resp.status_code}  {_health_resp.json()}")

_info_resp = client.get("/model-info")
assert _info_resp.status_code == 200, f"/model-info returned {_info_resp.status_code}"
print(f"GET /model-info -> {_info_resp.status_code}  {json.dumps(_info_resp.json())[:200]}...")

_sample_idx = 0
SAMPLE_CUSTOMER_ID = str(holdout_customer_ids[_sample_idx])
EXPECTED_EARLY_WARNING_SCORE = int(EARLY_WARNING_SCORE[_sample_idx])

_sample_history = (
    pl.scan_csv(str(RAW_TRAIN_DATA_PATH), schema_overrides={"customer_ID": pl.Utf8, "S_2": pl.Utf8})
    .filter(pl.col("customer_ID") == SAMPLE_CUSTOMER_ID)
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .sort("S_2")
    .select(["customer_ID"] + CANDIDATE_FEATURES)
    .collect(engine="streaming")
)
_sample_statements = []
for _row in _sample_history.iter_rows(named=True):
    _stmt = {}
    for _feat in CANDIDATE_FEATURES:
        _val = _row[_feat]
        _stmt[_feat] = None if (_val is None or (isinstance(_val, float) and np.isnan(_val))) else float(_val)
    _sample_statements.append(_stmt)
print(f"Sample customer {SAMPLE_CUSTOMER_ID}: {len(_sample_statements)} real statements pulled fresh from raw CSV")

_score_resp = client.post(
    "/score", json={"customer_id": SAMPLE_CUSTOMER_ID, "statements": _sample_statements}
)
assert _score_resp.status_code == 200, f"/score returned {_score_resp.status_code}: {_score_resp.text}"
_api_result = _score_resp.json()
_api_score = _api_result["early_warning_score"]
print(f"POST /score     -> {_score_resp.status_code}  early_warning_score={_api_score}, alert={_api_result['alert']}")

print(f"\nEnd-to-end check: API EARLY_WARNING_SCORE ({_api_score}) vs. this notebook's precomputed value "
      f"for the same customer ({EXPECTED_EARLY_WARNING_SCORE})")

API_SELF_TEST_PASSED = _api_score == EXPECTED_EARLY_WARNING_SCORE
if API_SELF_TEST_PASSED:
    print("\n✅ MATCH -- the live API's baseline/z-score computation is verified consistent with this "
          "notebook's direct computation on the same real customer.")
else:
    print("\n❌ MISMATCH -- do not deploy real_time_alert_service.py until this is resolved.")

if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 44's API self-test FAILED -- see ❌ line above. Not safe to proceed.")
print("\n✅ Section 17 complete.")


# =============================================================================
# SECTION 18: API LATENCY BENCHMARK
# =============================================================================
_section("SECTION 18: API Latency Benchmark")

_latency_payload = {"customer_id": SAMPLE_CUSTOMER_ID, "statements": _sample_statements}
N_API_LATENCY_SAMPLES = 150
_api_latencies_ms = []
for _ in range(N_API_LATENCY_SAMPLES):
    _t0 = time.perf_counter()
    _ = client.post("/score", json=_latency_payload)
    _api_latencies_ms.append((time.perf_counter() - _t0) * 1000.0)
_api_latencies_ms = np.array(_api_latencies_ms)
api_latency_summary = {
    "n_samples": N_API_LATENCY_SAMPLES,
    "p50_ms": round(float(np.percentile(_api_latencies_ms, 50)), 3),
    "p95_ms": round(float(np.percentile(_api_latencies_ms, 95)), 3),
    "p99_ms": round(float(np.percentile(_api_latencies_ms, 99)), 3),
    "max_ms": round(float(_api_latencies_ms.max()), 3),
}
print(f"/score latency over {N_API_LATENCY_SAMPLES} real TestClient calls: {api_latency_summary}")
print("\n✅ Section 18 complete.")


# =============================================================================
# SECTION 19: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 19: Deployment Readiness Checklist")

deployment_readiness_rows = [
    {"dimension": "Notebook 43 reproduction (deterministic)", "status": "PASS" if _reproduction_matches else "FAIL"},
    {"dimension": "Winning candidate reproduction", "status": "PASS" if _winning_reproduction_matches else "FAIL"},
    {"dimension": "Full statistical validation (all checks)", "status": "PASS" if ALL_STAT_CHECKS_PASS else "FAIL"},
    {"dimension": "Default-rate lift KPI (Notebook 42 target)", "status": "MET" if MEETS_KPI else "NOT MET"},
    {"dimension": "Deployment policy artifact persisted", "status": "PASS" if deployment_policy_path.exists() else "FAIL"},
    {"dimension": "API self-test (live, generated service, real customer)", "status": "PASS" if API_SELF_TEST_PASSED else "FAIL"},
    {"dimension": "API p99 latency < 500ms", "status": "PASS" if api_latency_summary["p99_ms"] < 500 else "FAIL"},
    {"dimension": "Overall recommendation",
     "status": "RECOMMENDED FOR PRODUCTION" if MEETS_KPI and ALL_STAT_CHECKS_PASS else "NOT RECOMMENDED FOR PRODUCTION"},
]
deployment_readiness_df = pd.DataFrame(deployment_readiness_rows)
deployment_readiness_path = EWS_DEPLOYMENT_DIR / "deployment_readiness_checklist.csv"
deployment_readiness_df.to_csv(deployment_readiness_path, index=False)
print(deployment_readiness_df.to_string(index=False))
print(f"✅ Saved -> {deployment_readiness_path}")
print("\n✅ Section 19 complete.")


# =============================================================================
# SECTION 20: CHARTS
# =============================================================================
_section("SECTION 20: Charts")

CHARTS_DIR = EWS_DEPLOYMENT_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(_valid_lift_boots, bins=40, color="#2563eb", alpha=0.75)
ax.axvline(winning_metrics["default_rate_lift"] or 0.0, color="#16a34a", linewidth=2,
           label=f"Point estimate ({(winning_metrics['default_rate_lift'] or 0.0):.2f}x)")
ax.axvline(LIFT_CI_LOWER, color="#dc2626", linestyle="--", linewidth=1.5, label=f"95% CI [{LIFT_CI_LOWER:.2f}x, {LIFT_CI_UPPER:.2f}x]")
ax.axvline(LIFT_CI_UPPER, color="#dc2626", linestyle="--", linewidth=1.5)
ax.axvline(EWS_KPI_TARGETS["min_default_rate_lift"], color="#7c3aed", linestyle=":", linewidth=1.5,
           label=f"KPI target ({EWS_KPI_TARGETS['min_default_rate_lift']}x)")
ax.set_xlabel("Bootstrap default-rate lift")
ax.set_ylabel("Resample count")
ax.set_title(f"Bootstrap Distribution -- Default-Rate Lift @ Candidate={WINNING_MIN_DEVIATION_COUNT}\n"
             f"({N_BOOTSTRAP:,} resamples)", fontsize=11)
ax.legend(fontsize=8)
_style_axes(ax)
chart1_path = CHARTS_DIR / "notebook_44_bootstrap_lift_distribution.png"
fig.tight_layout()
fig.savefig(chart1_path, dpi=150)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(calibration_table["bin"].astype(str), calibration_table["observed_default_rate"], color="#2563eb", alpha=0.8,
       label="Observed default rate")
ax.axhline(reproduced_base_default_rate, color="#64748b", linestyle="--", label=f"Base rate ({reproduced_base_default_rate:.3f})")
ax.set_xlabel("Normalized EARLY_WARNING_SCORE bin (low -> high)")
ax.set_ylabel("Observed default rate")
ax.set_title(f"Score-Rank Calibration -- Real Holdout Bins\n(monotonic: {CALIBRATION_MONOTONIC})", fontsize=11)
ax.legend(fontsize=8)
_style_axes(ax)
chart2_path = CHARTS_DIR / "notebook_44_calibration_by_score_bin.png"
fig.tight_layout()
fig.savefig(chart2_path, dpi=150)
plt.show()
plt.close(fig)

print(f"✅ Saved -> {chart1_path}")
print(f"✅ Saved -> {chart2_path}")
print(f"(Reusing Notebook 43's ROC/PR/lift-by-candidate charts in the report below -- not regenerated: "
      f"{NB43_ROC_CHART_PATH.name}, {NB43_PR_CHART_PATH.name}, {NB43_LIFT_CHART_PATH.name})")
print("\n✅ Section 20 complete.")


# =============================================================================
# SECTION 21: WORD REPORT
# =============================================================================
_section("SECTION 21: Word Report -- Early_Warning_Validation_Deployment_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 3, Problem 7: Early Warning System -- Validation & Deployment Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Scope & Selected Alert Threshold", level=1)
doc.add_paragraph(
    f"Statistical validation and deployment packaging for the rolling z-score early warning technique at "
    f"MIN_DEVIATION_COUNT={WINNING_MIN_DEVIATION_COUNT}, selected from Notebook 43's real default-rate-lift "
    f"sweep across candidates {sorted(CANDIDATE_RESULTS.keys())}. "
    + ("This candidate meets Notebook 42's default-rate lift KPI." if MEETS_KPI else
       "IMPORTANT: none of the tested candidates met Notebook 42's default-rate lift KPI on this real run "
       "-- this is the best-performing candidate (highest real lift), packaged for completeness, and is NOT "
       "RECOMMENDED FOR PRODUCTION until a future run finds a viable candidate.")
)

doc.add_heading("2. Full Classification Metrics-Suite Validation Summary", level=1)
doc.add_paragraph(
    "Per the platform's standing metrics-suite directive (effective Problem 6 onward): every metric below "
    "is a real, measured value reproduced independently in this notebook, cross-checked against Notebook "
    "43's originally-reported numbers (see Section 7-8's reproduction check)."
)
_t = doc.add_table(rows=1, cols=len(statistical_validation_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(statistical_validation_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in statistical_validation_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(statistical_validation_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Honest Limitation -- Deployment Scope & Assumptions", level=1)
for _k, _v in DEPLOYMENT_LIMITATION.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("4. Deployment Readiness Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(deployment_readiness_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(deployment_readiness_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in deployment_readiness_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(deployment_readiness_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("5. API Performance", level=1)
doc.add_paragraph(f"Latency over {api_latency_summary['n_samples']} real TestClient calls to /score (real "
                   f"customer statement history payload): p50={api_latency_summary['p50_ms']}ms, "
                   f"p95={api_latency_summary['p95_ms']}ms, p99={api_latency_summary['p99_ms']}ms, "
                   f"max={api_latency_summary['max_ms']}ms.")

doc.add_heading("6. Charts", level=1)
_chart_entries = [
    (chart1_path, f"Bootstrap distribution of default-rate lift at the selected candidate ({N_BOOTSTRAP:,} resamples)"),
    (chart2_path, "Score-rank calibration -- real holdout bins"),
]
if NB43_ROC_CHART_PATH.exists():
    _chart_entries.append((NB43_ROC_CHART_PATH, "ROC curve of the continuous EARLY_WARNING_SCORE (from Notebook 43)"))
if NB43_PR_CHART_PATH.exists():
    _chart_entries.append((NB43_PR_CHART_PATH, "Precision-Recall curve of the continuous EARLY_WARNING_SCORE (from Notebook 43)"))
if NB43_LIFT_CHART_PATH.exists():
    _chart_entries.append((NB43_LIFT_CHART_PATH, "Real default-rate lift by alert-threshold candidate (from Notebook 43)"))
for _cp, _cap in _chart_entries:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap)
    _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = EWS_DEPLOYMENT_DIR / "Early_Warning_Validation_Deployment_Report.docx"
doc.save(report_path)
print(f"✅ Saved -> {report_path}")
print("\n✅ Section 21 complete.")


# =============================================================================
# SECTION 22: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 22: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Notebook 43 reproduction matches exactly (deterministic)", _reproduction_matches)
_all_checks_passed &= _check("Winning candidate's confusion matrix/lift matches Notebook 43 exactly",
                              _winning_reproduction_matches)
_all_checks_passed &= _check("Bootstrap lift CI is well-formed (lower <= point estimate <= upper)",
                              LIFT_CI_LOWER <= (winning_metrics["default_rate_lift"] or 0.0) <= LIFT_CI_UPPER)
_all_checks_passed &= _check("Bootstrap AUC CI is well-formed (lower <= point estimate <= upper)",
                              AUC_CI_LOWER <= reproduced_auc <= AUC_CI_UPPER)
_all_checks_passed &= _check("Calibration table has at most 5 bins", 1 <= len(calibration_table) <= 5)
_all_checks_passed &= _check(
    "Winning candidate's confusion matrix sums to the real holdout customer count",
    sum(winning_metrics["confusion_matrix"].values()) == len(y_holdout),
)
_all_checks_passed &= _check("API self-test passed (real customer, live-imported service)", API_SELF_TEST_PASSED)
_expected_files = [
    statistical_validation_path, deployment_readiness_path, deployment_policy_path,
    service_py_path, env_example_path, requirements_api_path, chart1_path, chart2_path, report_path,
]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 22 complete -- all checks passed.")


# =============================================================================
# SECTION 23: WRITE NOTEBOOK 44 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 23: Write Notebook 44 Summary Artifact")

NB44_SUMMARY = {
    "notebook": "44_early_warning_system_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "winning_min_deviation_count": int(WINNING_MIN_DEVIATION_COUNT),
    "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "winning_candidate_metrics": winning_metrics,
    "bootstrap_lift_ci": [float(LIFT_CI_LOWER), float(LIFT_CI_UPPER)],
    "bootstrap_auc_ci": [float(AUC_CI_LOWER), float(AUC_CI_UPPER)],
    "bootstrap_pr_auc_ci": [float(PR_AUC_CI_LOWER), float(PR_AUC_CI_UPPER)],
    "calibration_monotonic": CALIBRATION_MONOTONIC,
    "split_half_score_psi": SCORE_PSI_SPLIT_HALF,
    "deployment_policy_path": str(deployment_policy_path),
    "service_py_path": str(service_py_path),
    "report_path": str(report_path),
    "statistical_validation_path": str(statistical_validation_path),
    "api_latency_summary": api_latency_summary,
    "random_seed": RANDOM_SEED,
}
NB44_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_44_summary.json"
with open(NB44_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB44_SUMMARY, f, indent=2)
print(f"Wrote: {NB44_SUMMARY_PATH}")

_section("NOTEBOOK 44 COMPLETE")
print(f"Winning MIN_DEVIATION_COUNT : {WINNING_MIN_DEVIATION_COUNT}")
print(f"Meets KPI target             : {MEETS_KPI}")
print(f"Recommended for production   : {NB44_SUMMARY['recommended_for_production']}")
print(f"Default-rate lift (reproduced): {(winning_metrics['default_rate_lift'] or 0.0):.3f}x "
      f"(95% CI [{LIFT_CI_LOWER:.3f}x, {LIFT_CI_UPPER:.3f}x])")
print(f"Secondary ROC-AUC            : {reproduced_auc:.4f}  (95% CI [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}])")
print(f"API self-test                : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Word report                  : {report_path}")
print(
    "\nNext: 45_early_warning_system_financial_impact_reporting_packaging.ipynb -- financial-impact "
    "reporting and final packaging for Problem 7, under the elevated Word/HTML reporting standard "
    "(synthesizing Notebooks 42-44's real content, not just this notebook's own financial figures), "
    "closing out Problem 7 of Phase 3."
)
